# A3 · Corrección telúrica

**Spec:** [`docs/spec_A3_v2_codex_telluric.md`](../docs/spec_A3_v2_codex_telluric.md)  |  **Bloque:** A · Reducción  |  **Run de este set:** `ROXs12b_realigned`

Corrige absorción telúrica para producir `cube_telcorr.fits`.

| | |
|---|---|
| **Entrada** | Cubo (post-cielo) |
| **Salida (QC/productos)** | `cube_telcorr.fits` |
| **Consume aguas abajo** | A4, B1 |


## Qué es la corrección telúrica y por qué STD_TELLURIC (no molecfit)

La atmósfera terrestre imprime **bandas de absorción** (O₂, H₂O) sobre el espectro, sobre todo en el rojo (>6800 Å): O₂ B ~6870 Å, la fuerte O₂ A ~7600 Å, y H₂O en ~7200/8200/9300 Å. **No son astrofísicas** — para recuperar la forma real del continuo (del compañero, muy rojo) hay que **dividir por la transmisión atmosférica**.

Dos caminos:
- **molecfit** — ajuste de un modelo físico de la atmósfera (lo preferido).
- **STD_TELLURIC** — la transmisión *observada* en la estrella estándar, escalada a la masa de aire de la ciencia por Beer–Lambert: `T_sci(λ) = T_std(λ)^(X_sci/X_std)`.

**Método adoptado = STD_TELLURIC** (nativo del DRS MUSE; ver `a3_telluric_justification.md`).

**Sobre molecfit (actualizado A1a, 2026-07-19):** el primer intento (2026-07-06) NO convergió (χ² congelado, transmisión→0). El reintento dedicado **A1a** halló la causa raíz real: **no era el GDAS** (secundario; se usó el perfil MIPAS estándar) sino que el espectro 1D se pasó **sin normalizar** (flujo mediano ~58000, continuo atascado en 1.0) y con **una sola ventana débil** (la banda B de O₂, 5.6% de absorción), dejando a O₂ **sin apalancamiento**. Al **normalizar el flujo** e incluir la **banda A de O₂ (7590–7690 Å, 30% de absorción)**, molecfit **converge** (`rel_col_O2=0.966±0.016`, `ppmv_O2≈205000` ≈20.5%, físico) y su transmisión **corrobora** STD_TELLURIC al **0.8%/píxel** en la banda B junto a Hα (tras alinear el marco vacío→aire de molecfit). Es decir: STD_TELLURIC no fue un atajo por fallo de molecfit — es el método del DRS, ahora con un **contraste independiente** que lo valida. Registrado en `a1a_molecfit_crosscheck` (stage00r_qc.json) y `a3_telluric_justification.md §6`.

**Ventanas protegidas (`T ≡ 1`, la corrección NO se aplica):**
- **6540–6590 Å** = Hα del compañero (diagnóstico de acreción).
- **5780–6050 Å** = láser AO de NFM.

**Consecuencia clave:** Hα está esencialmente libre de telúricas y su ventana está protegida → **A3 NUNCA afecta el resultado científico** (el límite de Hα). A3 solo importa para la fidelidad del **continuo rojo** que usan D1 y el modelado.


## Cómo ejecutar de forma independiente

Etapa de **reducción**: la celda de abajo resuelve el comando real para **este objeto** a partir de su `chain.reduction_profile` y de su config, y puede lanzarlo. Son trabajos largos (ver coste), así que se lanzan en segundo plano con el log a la vista; el notebook no se bloquea.

Si algún dato no está declarado en el config del run, la celda lo dice y **no lanza** en vez de inventarse una ruta.

Comando histórico de referencia:

```bash
conda activate MUSE
bash scripts/telluric.sh
```


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))


## Ejecutar o auditar


In [ ]:
cmd, target_run, missing = nb.launch_command('A3', RUN_ID)
print('run que ejecuta esta etapa:', target_run)
print('comando resuelto para este objeto:')
print('   ', cmd or '(sin plantilla)')
if missing:
    print()
    print('NO se puede lanzar: faltan datos en el config del run.')
    print('   sin resolver:', ', '.join(missing))
    print(f'   declara esas claves en runs/{target_run}/config/config.json')

RUN = False   # -> True para LANZAR (trabajo largo: revisa el coste arriba)

if RUN and not missing:
    import subprocess, time
    from pathlib import Path
    log = Path(nb.run_dir(target_run)) / 'logs' / f'a3_launch.log'
    log.parent.mkdir(parents=True, exist_ok=True)
    with open(log, 'w') as fh:
        proc = subprocess.Popen(cmd, shell=True, cwd=str(nb.project_root()),
                                stdout=fh, stderr=subprocess.STDOUT)
    print(f'lanzado en segundo plano (pid {proc.pid}); log -> {log}')
    print('sigue el progreso con:  !tail -f', log)
elif RUN:
    print('RUN=True pero hay datos sin resolver: no se lanza nada.')
else:
    print()
    print('Modo auditoría (RUN=False): abajo se carga el QC existente.')


## Resultados que llevaron a la conclusión

La celda lee el `stage00t_qc.json` **de la cadena de este objeto** y no supone su esquema, porque hay **dos en circulación** y no comparten campos: el de las reducciones antiguas (`threshold_pct`, `method`, escala por `airmass_*`, V1 pre→post) y el de la cadena multi-noche (`checkpoint_required`, ajuste por `molecules`/`regions_A`, V1 como residuo por banda). Se imprime el que traiga el QC, y **un campo que no exista se dice**, no se deja en blanco — indexar a ciegas es lo que rompía esta celda con `KeyError: 'threshold_pct'`.

Y ojo con leer las V como si siempre validaran una corrección: si `telluric_applied = False`, V1/V2/V4 confirman que **no se tocó nada**, no que se corrigiera bien. La celda lo avisa cuando toca.


In [ ]:
qt = nb.load_qc_optional('stages/stage00t_qc.json', RUN_ID)
print()
if qt is None:
    print('A3 no emitió stage00t_qc.json en esta cadena: nada que auditar aquí.')
    print("Declara el run que lo contiene en chain.stage_runs['A3'], o ejecuta la etapa.")
else:
    # Hay DOS esquemas de A3 en circulación y NO comparten todos los campos:
    #   · reducciones antiguas: decision.threshold_pct/method, fit.airmass_*,
    #     verification.v1_o2_depth_pre_post_pct / v2_outside_bands_unchanged
    #   · cadena multi-noche:   decision.checkpoint_required, fit.molecules/regions_A,
    #     verification.v1_residual_pct_by_band / v2_outside_bands_change_pct
    # Se imprime el que traiga el QC. Un campo ausente se DICE (no se deja en
    # blanco ni se inventa): indexar a ciegas es lo que rompía esta celda.
    d = qt.get('decision') or {}
    fit = qt.get('fit') or {}
    ver = qt.get('verification') or {}

    def _f(dd, key, unit=''):
        if key not in dd:
            return '(no está en el esquema de este QC)'
        v = dd[key]
        return 'n/d' if v is None or v == '' else f'{v}{unit}'

    print('Decisión:', _f(d, 'verdict'), '| aplicado:', _f(d, 'telluric_applied'),
          '| checkpoint:', _f(d, 'user_checkpoint'))
    print('  umbral:', _f(d, 'threshold_pct', ' %'), '| método:', _f(d, 'method'))
    _dp = d.get('depth_pct_by_band') or {}
    if _dp:
        _mx = float(d.get('max_depth_pct', max(_dp.values())))
        print('  profundidad por banda [%]:',
              ', '.join(f'{k}={float(v):.3f}' for k, v in _dp.items()), f'(máx {_mx:.3f})')
    print()

    if 'airmass_std' in fit:      # esquema antiguo: escala por masa de aire
        _xs, _xc = fit.get('airmass_std'), fit.get('airmass_sci')
        _r = f'{_xc / _xs:.3f}' if (_xs and _xc) else 'n/d'
        print(f"Escala airmass: X_std={_xs} -> X_sci={_xc}  ({fit.get('scaling')} = {_r})")
        print('  fuente:', fit.get('source', 'n/d'))
    else:                          # esquema multi-noche: ajuste por moléculas
        print('Ajuste:', ', '.join(fit.get('molecules') or []) or 'n/d',
              '| regiones [Å]:', fit.get('regions_A'))
        print('  kernel:', fit.get('kernel'), '| chi2 por región:',
              fit.get('chi2_by_region') or '(vacío: no se ajustó nada)')
    print()

    print('Verificación:')
    if 'v1_o2_depth_pre_post_pct' in ver:
        _pp = ver['v1_o2_depth_pre_post_pct']
        print(f'  V1 O2 B (~6870 A) pre -> post: {_pp[0]}% -> {_pp[1]}%')
        print('  V2 fuera de bandas sin cambio:', ver.get('v2_outside_bands_unchanged'))
    else:
        print('  V1 residuo por banda [%]:',
              ver.get('v1_residual_pct_by_band') or '(vacío: no se aplicó corrección)')
        print('  V2 cambio fuera de bandas [%]:', ver.get('v2_outside_bands_change_pct'))
    print('  V3 Halpha intacta:', _f(ver, 'v3_halpha_untouched'),
          '| V4 transmisión física [0,1]:', _f(ver, 'v4_transmission_physical'),
          '| V5 STAT escalado:', _f(ver, 'v5_stat_scaled'))
    if not d.get('telluric_applied'):
        print()
        print('  OJO: telluric_applied = False -> V1/V2/V4 no verifican una corrección')
        print('  aplicada, sino que NO se tocó nada. La decisión está en la figura de abajo.')
print()
print('molecfit (crosscheck A1a, si el A1 de esta cadena lo trae):')
try:
    qr = nb.load_qc('stages/stage00r_qc.json', RUN_ID)
    cc = qr['a1a_molecfit_crosscheck']
    mf = cc['model_fit']; tcB = cc['transmission_comparison_vs_std_telluric']['O2_B_band_6864_6960A']
    print(f"  converge: mpfit status={mf['mpfit_status']}, rel_col_O2={mf['rel_mol_col_O2']}+-{mf['rel_mol_col_O2_unc']}, ppmv_O2={mf['ppmv_O2']:.0f}")
    print(f"  vs STD_TELLURIC en banda B (junto a Halpha): |dT|/px={tcB['mean_abs_dT_per_pixel']}, razon absorcion={tcB['integrated_absorption_ratio_molecfit_over_std']}")
    print('  =>', cc['conclusion'][:110], '...')
except (FileNotFoundError, KeyError) as e:
    print('  (no disponible para esta cadena:', type(e).__name__, e, ')')


## De dónde sale la corrección — o por qué no se aplicó

La celda **no busca ficheros en disco**: lee el QC de A3 (`stages/stage00t_qc.json`), que es quien declara en `products.transmission` si la etapa emitió la curva y en `decision` si concluyó que no hacía falta aplicarla. Según eso hace una cosa u otra:

- **A3 aplicó corrección** → grafica `TELLURIC_TRANS.fits` (transmisión 1D derivada de `STD_TELLURIC_0001.fits` de `muse_standard`, escalada a la masa de aire de la ciencia; archivo pequeño, no hace falta el cubo): transmisión vs λ con las bandas telúricas (O₂ B ~6870, la fuerte O₂ A ~7600, H₂O ~7200/8200/9300) y las **ventanas protegidas** (gris, `T ≡ 1`) — la de Hα (6540–6590) queda plana justo antes de O₂ B.
- **A3 decidió que no hacía falta** → no hay curva que graficar, y eso **no es un fallo**: grafica la medida que llevó a esa decisión, la profundidad de cada banda frente al umbral del 3 %.

Solo si el QC **no declara** la ruta (QCs antiguos) prueba los dos sitios previstos, siempre dentro del propio run. Y si A3 dice `telluric_applied = True` pero la curva no aparece, **falla ruidosamente** en vez de graficar otra cosa.

> Ojo al comparar con números de otras reducciones: el veredicto de A3 es **por cubo**. La cadena canónica de ROXs 12 b da `not_needed_shallow` con O₂ B **0.59 %**, mientras la primera auto-reducción daba **6.76 %** y su realineado **7.34 %**, ambos con corrección aplicada. Las tres decisiones están desdobladas más abajo; manda el QC que imprime esta celda. **La diferencia entre esos números ya está explicada** (el DRS corrige el telúrico por exposición en la cadena multi-noche y no lo hacía en las antiguas): la medida está en `debug/A3_telluric_debug.ipynb` y el resumen, en «Decisiones y notas».


In [ ]:
MAKE_PLOT = True   # archivo pequeño; requiere kernel MUSE (astropy)
if MAKE_PLOT:
    try:
        import numpy as np
        import matplotlib.pyplot as plt
        from pathlib import Path
        from astropy.io import fits

        def _anchor(value):
            """Ruta del QC -> absoluta. Unas vienen absolutas y otras RELATIVAS a la
            raíz del repo, y el cwd del notebook es notebooks/<objeto>/ (la celda de
            setup añade la raíz al sys.path pero NO hace chdir)."""
            p = Path(value)
            return p if p.is_absolute() else nb.project_root() / p

        # La fuente de verdad es el QC de A3, no una búsqueda de ficheros: la etapa
        # declara en products.transmission si emitió la curva, y en decision si
        # decidió que no hacía falta aplicar nada.
        qc3 = nb.load_qc_optional('stages/stage00t_qc.json', RUN_ID)
        if qc3 is None:
            print('A3 no ha corrido para este objeto: no hay stage00t_qc.json en su cadena.')
            print('   (nada que graficar; la decisión de abajo aún no está registrada)')
        else:
            dec = qc3.get('decision', {})
            declared = (qc3.get('products', {}) or {}).get('transmission') or ''
            applied = dec.get('telluric_applied')
            print(f"A3: telluric_applied = {applied}   verdict = {dec.get('verdict')}")

            trans_path = None
            if declared:
                trans_path = _anchor(declared)
                print('curva declarada por el QC:', trans_path)
                if not trans_path.exists():
                    raise FileNotFoundError(
                        f'el QC declara la transmisión en {trans_path} y no está en disco')
            else:
                # QCs antiguos no persistían la ruta: se prueban los dos sitios
                # previstos, SIEMPRE dentro del propio run (nunca el de otro).
                cands = []
                cube_path = qc3.get('input', {}).get('cube')
                if cube_path:
                    cands.append(('junto al cubo de A1',
                                  _anchor(cube_path).parent / 'TELLURIC_TRANS.fits'))
                cands.append(('raw_reduction/ del run',
                              nb.run_dir(RUN_ID) / 'raw_reduction' / 'TELLURIC_TRANS.fits'))
                for _lab, _p in cands:
                    if _p.exists():
                        trans_path = _p
                        print(f'el QC no declara la curva; encontrada ({_lab}):', _p)
                        break
                if trans_path is None and applied:
                    raise FileNotFoundError(
                        'A3 dice telluric_applied=True pero no hay curva ni declarada ni en disco -> '
                        + ' | '.join(f'{l}: {p}' for l, p in cands))

            if trans_path is not None:
                h = fits.open(trans_path); t = h[1].data
                wave = np.asarray(t['wave_A'], dtype=float)
                trans = np.asarray(t['transmission'], dtype=float)
                print('FITS usado:', trans_path)

                fig, ax = plt.subplots(figsize=(11, 4))
                ax.plot(wave, trans, lw=1.1, color='tab:blue')
                for i, (a, b) in enumerate(qc3.get('protected_windows_A') or [(6540, 6590), (5780, 6050)]):
                    ax.axvspan(a, b, color='0.6', alpha=0.35,
                               label='ventana protegida (T≡1)' if i == 0 else None)
                for x, lab in [(6870, 'O₂ B'), (7200, 'H₂O'), (8200, 'H₂O'), (9300, 'H₂O')]:
                    ax.annotate(lab, (x, np.interp(x, wave, trans)), textcoords='offset points',
                                xytext=(0, -14), ha='center', fontsize=8, color='tab:red')
                ax.set_xlabel('λ [Å]'); ax.set_ylabel('Transmisión telúrica aplicada')
                ax.set_title('A3 · TELLURIC_TRANS.fits (STD_TELLURIC escalado a airmass sci)')
                ax.set_ylim(0, 1.05); ax.legend(fontsize=8); fig.tight_layout()
                out = 'transmission.png'
                h.close()
            else:
                # No hay curva porque A3 decidió que no hacía falta. Eso NO es un
                # fallo: se grafica la medida que llevó a esa decisión.
                depths = dec.get('depth_pct_by_band') or {}
                print('A3 no emitió curva de transmisión: decidió que no hacía falta aplicarla.')
                print('   profundidad medida por banda [%]:',
                      ', '.join(f'{k}={v:.3f}' for k, v in depths.items()) or '(no registrada)')
                if not depths:
                    raise RuntimeError('el QC de A3 no registra depth_pct_by_band: nada que graficar')
                _k = list(depths); _v = [float(depths[k]) for k in _k]
                fig, ax = plt.subplots(figsize=(7.5, 4))
                ax.bar(range(len(_k)), _v, color=['tab:red' if x > 3 else 'tab:green' for x in _v],
                       alpha=0.85)
                ax.axhline(3.0, color='tab:red', ls='--', lw=1.2,
                           label='umbral de la etapa (3 %)')
                ax.set_xticks(range(len(_k))); ax.set_xticklabels(_k, fontsize=9)
                ax.set_ylabel('profundidad de la banda [%]')
                ax.set_ylim(0, max(3.6, max(_v) * 1.25))
                ax.set_title(f"A3 · por qué NO se aplicó: {dec.get('verdict')}\n"
                             'profundidad medida frente al umbral', fontsize=10)
                ax.legend(fontsize=8); fig.tight_layout()
                out = 'depth_vs_threshold.png'

            outdir = nb.run_dir(RUN_ID) / 'plots' / 'a3_telluric'
            outdir.mkdir(parents=True, exist_ok=True)
            fig.savefig(outdir / out, dpi=110)
            print('figura ->', outdir / out)
            plt.show()
    except Exception as e:
        print('No se pudo generar el plot:', type(e).__name__, e)
        print('Necesita el kernel MUSE (astropy). La curva, si A3 la emitió, sale de '
              'products.transmission de su QC.')


## Decisiones y notas
- **STD_TELLURIC + escala por airmass** (método nativo del DRS MUSE). El reintento A1a hizo **converger** molecfit (causa raíz previa: flujo sin normalizar + banda débil, no el GDAS) y su transmisión **corrobora** STD_TELLURIC al 0.8%/px en la banda B junto a Hα. · [`docs/a3_telluric_justification.md`](../docs/a3_telluric_justification.md)
- Etapa **condicional**, y el veredicto es **por cubo**: no hay una sola decisión de A3, sino una por reducción. **La que manda en este notebook** es la de su propia cadena: `verdict = not_needed_shallow`, `telluric_applied = False`, profundidad O₂ B = 0.586 % frente al umbral del 3 %. Si sale `n/d`, A3 no ha corrido para esta cadena y no hay decisión registrada para el objeto.
- Histórico de ROXs 12 b, desdoblado para que no se citen números de un cubo en otro — las tres tienen QC propio y las tres dicen cosas distintas: **(a) `ROXs12b_raw`**, primera auto-reducción (`muse_scipost`), `stages/stage00t_qc.json` de ese run: O₂ B **6.76 %**, H₂O 7200 0.658 % → `needed`, **aplicado**, checkpoint humano **aprobado**. **(b) El realineado de esa misma reducción** (`muse_scipost_aligned`, alias `stages/stage00t_realigned_qc.json`): O₂ B **7.34 %**, H₂O 7200 1.0 %, H₂O 8200 0.054 % → `needed`, **aplicado**, sin checkpoint registrado; su curva es la que vive en `/mnt/2TB/MUSE_work/ROXs12b_realigned/TELLURIC_TRANS.fits`, **fuera** del run. **(c) Multi-noche del 2026-07-28** (la cadena canónica de hoy): O₂ B **0.586 %** → `not_needed_shallow`, **no aplicado**, sin curva emitida.
- **Cerrado (2026-07-30):** que la banda O₂ B pase de ~7 % en las reducciones antiguas a 0.59 % en la multi-noche **no es una banda más superficial: es el residuo de una corrección ya aplicada**. La cadena multi-noche le pasa `STD_TELLURIC` a `muse_scipost` en cada una de las 29 exposiciones —lo exige `musepipe/reduction/perexp_plan.py`, que aborta si no hay exactamente una por exposición— y los SOF de las reducciones antiguas **no lo llevaban** (16 SOF con `STD_RESPONSE` y cero `STD_TELLURIC`). La prueba independiente es **O₂ A (7590–7700 Å)**, que A3 no mide y que cae de **28.8 %** a **1.25 %** entre una reducción y la otra. La masa de aire no lo explica: con las 29 exposiciones reales (X = 1.02–1.54, peso `EXPTIME`) Beer–Lambert da un factor **1.03×**, y del signo contrario. Luego la corrección telúrica del cubo canónico **está hecha, por el DRS, exposición a exposición y cada una a su propia masa de aire** — mejor que aplicar una curva única al cubo ya combinado. Medida y trazada en `notebooks/<objeto>/debug/A3_telluric_debug.ipynb`.


## Conclusión (registrada)

**En la cadena de ESTE notebook: `telluric_applied = False`, `verdict = not_needed_shallow`** (O₂ B = 0.586 % frente al umbral del 3 %). `n/d` = A3 no ha corrido para este objeto.

Lo de abajo es la conclusión **de la reducción de referencia** (`ROXs12b_raw`), donde sí se aplicó: se conserva porque es la que justifica el método para el paper, pero **no describe el cubo de la cadena multi-noche** — sus números son de otro cubo (ver el desdoble en «Decisiones y notas»).

**A3 en `ROXs12b_raw`: corrección telúrica APLICADA por STD_TELLURIC escalado a airmass (`telluric_applied = True`, `needed`, checkpoint aprobado).**

- **Fecha:** QC telúrico de referencia 2026-07-06 (`ROXs12b_raw`); justificación para paper 2026-07-10 (`docs/a3_telluric_justification.md`).
- **Datos:** `STD_TELLURIC_0001.fits` (muse_standard), escala airmass 1.087 → 1.158 (exp. 1.065); aplicado → `cube_telcorr.fits`; transmisión en `TELLURIC_TRANS.fits`.
- **Evidencia:** O₂ B 6.76% → 0.6%; continuo fuera de bandas sin cambio; Hα intacta (ventana protegida); STAT escala como T².
- **molecfit (A1a, 2026-07-19): CONVERGE y corrobora STD_TELLURIC.** La no-convergencia previa no era por GDAS sino por flujo sin normalizar + banda B débil (O₂ sin apalancamiento). Normalizando el flujo + banda A de O₂ → `rel_col_O2=0.966±0.016`, y T(λ) coincide con STD_TELLURIC al 0.8%/px en la banda B junto a Hα → contraste independiente. Residuo O₂ ~0.6% presupuestado para D2. Ver `a1a_molecfit_crosscheck` + `a3_telluric_justification.md §6`.
- **Corregido el 2026-07-30:** era falso que «el realineado no emitió su propio `stage00t_qc.json`». Sí lo emitió, y hay **tres** QCs de A3 para ROXs 12 b diciendo cosas distintas — están desdoblados en «Decisiones y notas». La cadena multi-noche además **no aplica** la corrección.
- **Impacto en Hα = nulo** (ventana protegida) → el límite de acreción no depende de esta corrección.
